# ARVI-RX - MedGemma 4B Colab demo

This notebook runs a real multimodal inference with `google/medgemma-4b-it` on Google Colab GPU.

It is an educational prototype only. It is not a medical device and must not be used for diagnosis, triage, or patient care.

## 0. Runtime requirement

Before running the notebook, select a GPU runtime:

`Runtime` -> `Change runtime type` -> `T4 GPU` or better.

You also need Hugging Face access to the gated model `google/medgemma-4b-it`.

In [ ]:
!pip -q install -U transformers accelerate pillow huggingface_hub torch

In [ ]:
import json
import re
from pathlib import Path
from getpass import getpass

import torch
from huggingface_hub import login
from PIL import Image
from google.colab import files
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "google/medgemma-4b-it"
PROMPT_VERSION = "medgemma_colab_strict_json_v1"
WARNING_TEXT = "Prototype pedagogique. Non destine au diagnostic. Validation par un professionnel qualifie requise."
ALLOWED_CLASSES = {"normal", "suspected_opacity", "uncertain"}

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("A Colab GPU runtime is required for this MedGemma 4B demo.")

## 1. Hugging Face login

Paste a Hugging Face token with access to `google/medgemma-4b-it`. The token is read with `getpass`, so it is not displayed in the notebook.

In [ ]:
hf_token = getpass("Hugging Face token: ")
login(token=hf_token)
del hf_token
print("Logged in to Hugging Face.")

## 2. Load MedGemma 4B

The model is loaded on GPU. On recent Colab GPUs, `bfloat16` is preferred when available; otherwise the notebook uses `float16`.

In [ ]:
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("Loading", MODEL_ID, "with dtype", dtype)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map={"": "cuda:0"},
)
model.eval()

has_meta = any(getattr(t, "is_meta", False) for t in list(model.parameters()) + list(model.buffers()))
if has_meta:
    raise RuntimeError("Model contains meta tensors; weights were not fully loaded.")

print("Model loaded on GPU.")

## 3. Prompt and JSON helpers

In [ ]:
STRICT_PROMPT = f"""
Tu es un assistant radiologue virtuel pedagogique pour le projet ARVI-RX.
Analyse cette radiographie thoracique frontale.

Reponds uniquement avec un JSON valide. N'ajoute aucun texte hors JSON.

Schema obligatoire :
{{
  "image_quality": "good" | "limited" | "poor",
  "predicted_class": "normal" | "suspected_opacity" | "uncertain",
  "confidence": nombre entre 0 et 1,
  "visual_evidence": ["observation visuelle prudente"],
  "justification": "justification courte et prudente",
  "limitations": ["limite explicite"],
  "warning": "{WARNING_TEXT}",
  "model_name": "{MODEL_ID}",
  "prompt_version": "{PROMPT_VERSION}"
}}

Contraintes :
- Ne donne pas de diagnostic medical definitif.
- Si l'image est ambigue, de mauvaise qualite ou si tu n'es pas sur, utilise "uncertain".
- Les seules classes autorisees sont normal, suspected_opacity, uncertain.
""".strip()


def extract_json_object(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        payload = json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
        if not match:
            raise
        payload = json.loads(match.group(0))
    if not isinstance(payload, dict):
        raise ValueError("Model response JSON is not an object")
    return payload


def normalize_prediction(payload: dict) -> dict:
    predicted_class = str(payload.get("predicted_class") or payload.get("class") or "uncertain")
    if predicted_class not in ALLOWED_CLASSES:
        predicted_class = "uncertain"
    try:
        confidence = float(payload.get("confidence", 0.0))
    except Exception:
        confidence = 0.0
    confidence = max(0.0, min(confidence, 1.0))

    evidence = payload.get("visual_evidence") or payload.get("observations") or []
    if isinstance(evidence, str):
        evidence = [evidence]
    if not isinstance(evidence, list):
        evidence = []

    limitations = payload.get("limitations") or payload.get("limits") or []
    if isinstance(limitations, str):
        limitations = [limitations]
    if not isinstance(limitations, list):
        limitations = []

    return {
        "image_quality": str(payload.get("image_quality") or "limited"),
        "predicted_class": predicted_class,
        "confidence": round(confidence, 3),
        "visual_evidence": [str(item) for item in evidence],
        "justification": str(payload.get("justification") or "No justification provided."),
        "limitations": [str(item) for item in limitations] or ["Educational prototype; professional review required."],
        "warning": WARNING_TEXT,
        "model_name": MODEL_ID,
        "prompt_version": PROMPT_VERSION,
    }


def run_medgemma_image(image_path: str | Path) -> dict:
    image = Image.open(image_path).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": STRICT_PROMPT},
            ],
        }
    ]
    prompt_text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=prompt_text, images=image, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    raw_text = processor.decode(generated[0][input_len:], skip_special_tokens=True).strip()
    payload = extract_json_object(raw_text)
    final_json = normalize_prediction(payload)
    final_json["raw_model_response"] = raw_text
    return final_json

## 4. Upload one PNG/JPG image

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No image uploaded.")

image_path = Path(next(iter(uploaded.keys())))
if image_path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
    raise ValueError("Please upload a PNG/JPG image.")

image = Image.open(image_path).convert("RGB")
display(image.resize((384, 384)))
print("uploaded:", image_path, image.size, image.mode)

## 5. Run real MedGemma inference and save JSON

In [ ]:
result = run_medgemma_image(image_path)
print(json.dumps(result, indent=2, ensure_ascii=False))

out_path = Path("medgemma_demo_result.json")
out_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print("saved:", out_path.resolve())
files.download(str(out_path))

## 6. Optional batch test on 3 to 5 images

Upload several PNG/JPG files. The notebook will run true MedGemma inference for the first 5 images and save `medgemma_batch_results.json`.

In [ ]:
# Optional cell
batch_upload = files.upload()
batch_paths = [Path(name) for name in batch_upload.keys() if Path(name).suffix.lower() in {".png", ".jpg", ".jpeg"}]
batch_paths = batch_paths[:5]

batch_results = []
for path in batch_paths:
    print("Running:", path)
    try:
        prediction = run_medgemma_image(path)
        prediction["image_path"] = str(path)
    except Exception as exc:
        prediction = {
            "image_path": str(path),
            "image_quality": "limited",
            "predicted_class": "uncertain",
            "confidence": 0.0,
            "visual_evidence": [],
            "justification": f"MedGemma batch inference failed: {exc}",
            "limitations": ["batch inference failure", "educational prototype"],
            "warning": WARNING_TEXT,
            "model_name": MODEL_ID,
            "prompt_version": PROMPT_VERSION,
        }
    batch_results.append(prediction)

print(json.dumps(batch_results, indent=2, ensure_ascii=False))
batch_out = Path("medgemma_batch_results.json")
batch_out.write_text(json.dumps(batch_results, indent=2, ensure_ascii=False), encoding="utf-8")
print("saved:", batch_out.resolve())
files.download(str(batch_out))